# Regression

In [80]:
seed = 42
np.random.seed(seed)

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import sklearn.linear_model as lm

sns.set_style('darkgrid')
sns.set_theme(font_scale=1.5)

df = pd.read_csv("data/heartDisease.csv")
df = df.drop(labels=["row.names"], axis=1)
famhist_le = LabelEncoder()
df["famhist_num"] = famhist_le.fit_transform(df["famhist"])

included_variables = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age", "chd", "famhist_num"]

X, y = df[included_variables], df["obesity"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
X_train


,sbp,tobacco,ldl,typea,alcohol,age,chd,famhist_num
411,166,0.80,5.63,50,28.80,60,0,0
265,118,0.12,1.96,37,2.42,18,0,0
57,126,5.10,2.96,55,12.34,38,1,0
199,148,0.00,4.66,50,4.03,27,0,0
175,122,4.00,5.24,45,0.00,61,1,1
...,...,...,...,...,...,...,...,...
106,108,1.50,4.33,66,21.60,61,1,0
270,130,0.00,4.16,46,0.00,55,1,1
348,130,0.08,5.59,50,6.27,43,1,1
435,136,0.00,1.77,45,2.06,16,0,0


# Which variables to pick?

All except **adiposity**

In [77]:
def preprocess(X, categorical_variables, continuous_variables):
    # Continuous

    scaler = StandardScaler()
    
    X_cont = X[continuous_variables].assign(
        tobacco = lambda x: np.log(x["tobacco"] + 1/10000),
        alcohol = lambda x: np.log(x["alcohol"] + 1/10000)
    )
    
    X_cont = pd.DataFrame(scaler.fit_transform(X_cont), columns=scaler.get_feature_names_out())
    
    # Categorical
    onehot = OneHotEncoder()

    X_cat = pd.DataFrame(onehot.fit_transform(X[categorical_variables]).toarray()
                 , columns=onehot.get_feature_names_out())
    X = X.drop(["chd", "famhist_num"], axis=1)


    return pd.merge(X_cont, X_cat, how='inner', right_index=True, left_index=True)


X_train_preprocessed = preprocess(X_train, categorical_variables=["chd", "famhist_num"], continuous_variables=["sbp", "tobacco", "ldl", "typea", "alcohol", "age"])


In [112]:
from sklearn.base import BaseEstimator, TransformerMixin


class LogTransformer(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return X + 100

In [116]:
from sklearn.compose import ColumnTransformer


num_pipeline = ColumnTransformer(transformers=[
    
    ('log_transform', LogTransformer(), ["alcohol", "tobacco"]),
], remainder="passthrough",verbose_feature_names_out=False).set_output(transform='pandas')


pd.DataFrame(num_pipeline.fit_transform(X_train), columns=X_train.columns)


ValueError: Unable to configure output for LogTransformer() because `set_output` is not available.

In [108]:
X_train

,sbp,tobacco,ldl,typea,alcohol,age,chd,famhist_num
411,166,0.80,5.63,50,28.80,60,0,0
265,118,0.12,1.96,37,2.42,18,0,0
57,126,5.10,2.96,55,12.34,38,1,0
199,148,0.00,4.66,50,4.03,27,0,0
175,122,4.00,5.24,45,0.00,61,1,1
...,...,...,...,...,...,...,...,...
106,108,1.50,4.33,66,21.60,61,1,0
270,130,0.00,4.16,46,0.00,55,1,1
348,130,0.08,5.59,50,6.27,43,1,1
435,136,0.00,1.77,45,2.06,16,0,0


In [ ]:

cat_vars = ["chd", "famhist_num"]
num_vars = ["sbp", "tobacco", "ldl", "typea", "alcohol", "age"]


num_pipeline = Pipeline([
    ('add_variables', NewVariablesAdder()),
    ('std_scaler', StandardScaler())
])


pipeline = ColumnTransformer([
    ('numerical', num_pipeline, num_vars),
    ('categorical', cat_pipeline, cat_vars),
])

airbnb_processed = data_pipeline.fit_transform(airbnb_data)

In [ ]:
lam = 0.1

model = lm.Ridge(alpha=lam)
model = model.fit(X_train_preprocessed, y_train)
y_est = model.predict(X_test)



TypeError: preprocess() missing 2 required positional arguments: 'categorical_variables' and 'continuous_variables'